# Quality és generált gyertya előállítás

Ez a notebook a bronze rétegben lévő brókeres OHLCV adatokat készíti elő quality vizsgálatra, majd calendar alapján generált gyertyákat hoz létre.

A folyamat:

1. Betölti a bronze adatokat Azure-ból.
2. Betölti a silver/calendar adatokat Azure-ból.
3. Az eredeti betöltött adatokat nem módosítja, csak memóriában dolgozik másolaton.
4. A timestamp mezőket percre igazítja, ezért a másodperc/milliszekundum eltérés nem okoz hibát.
5. Csak azokat az időpontokat tartja meg, amelyek szerepelnek a calendarban.
6. Brókerenként széles táblát épít: `binance_close`, `dukascopy_close`, `saxo_bank_close`, stb.
7. A brokerek adataiból generált OHLC gyertyát készít.

Jelenlegi generálási módszer:

- `method=0`: medián alapú OHLC generálás

A generált gyertya minőségi jelölése:

- `multi_source`: legalább két broker adott adatot
- `single_source`: csak egy broker adott adatot
- `missing`: calendar szerint kellett volna gyertya, de nincs broker adat

A notebook nem ír vissza Azure-ba, csak betölt és kimutatást/generált táblát készít.


In [1]:
import importlib

import quality.candle_generation

importlib.reload(quality.candle_generation)

from quality.candle_generation import build_generated_candles


START_MONTH = "2024-01"
END_MONTH = "2024-02"
BROKERS = None
ASSETS = None
INTERVAL = "1m"
GENERATION_METHOD = 0


generated_df = build_generated_candles(
    start_month=START_MONTH,
    end_month=END_MONTH,
    brokers=BROKERS,
    assets=ASSETS,
    interval=INTERVAL,
    method=GENERATION_METHOD,
    source="azure",
    print_progress=True,
)

generated_df

reading bronze/binance/btcusd/2024/01/BTCUSDT-1m-2024-01.parquet
reading bronze/binance/btcusd/2024/02/BTCUSDT-1m-2024-02.parquet
reading bronze/dukascopy/btcusd/2024/01/BTCUSD-1m-2024-01.parquet
reading bronze/dukascopy/btcusd/2024/02/BTCUSD-1m-2024-02.parquet
reading bronze/dukascopy/dax/2024/01/DEUIDXEUR-1m-2024-01.parquet
reading bronze/dukascopy/dax/2024/02/DEUIDXEUR-1m-2024-02.parquet
reading bronze/dukascopy/eurusd/2024/01/EURUSD-1m-2024-01.parquet
reading bronze/dukascopy/eurusd/2024/02/EURUSD-1m-2024-02.parquet
reading bronze/dukascopy/us500/2024/01/USA500IDXUSD-1m-2024-01.parquet
reading bronze/dukascopy/us500/2024/02/USA500IDXUSD-1m-2024-02.parquet
reading bronze/dukascopy/xagusd/2024/01/XAGUSD-1m-2024-01.parquet
reading bronze/dukascopy/xagusd/2024/02/XAGUSD-1m-2024-02.parquet
reading bronze/dukascopy/xauusd/2024/01/XAUUSD-1m-2024-01.parquet
reading bronze/dukascopy/xauusd/2024/02/XAUUSD-1m-2024-02.parquet
reading bronze/interactive_brokers/dax/2024/01/DAX-1m-2024-01.parque

,time,asset,year,month,expected,calendar_source,binance_open,dukascopy_open,interactive_brokers_open,saxo_bank_open,...,generated_candle_slot,generated_open,generated_high,generated_low,generated_close,generated_volume,generation_method_id,generation_method,consensus_quality,is_generated
0,2024-01-01 00:00:00+00:00,BTCUSD,2024,1,True,generated_24_7,42283.58,NaN,NaN,NaN,...,True,42283.580,42298.620,42261.0200,42298.610,<NA>,0,median_ohlc,single_source,True
1,2024-01-01 00:01:00+00:00,BTCUSD,2024,1,True,generated_24_7,42298.62,NaN,NaN,NaN,...,True,42298.620,42320.000,42298.6100,42320.000,<NA>,0,median_ohlc,single_source,True
2,2024-01-01 00:02:00+00:00,BTCUSD,2024,1,True,generated_24_7,42319.99,NaN,NaN,NaN,...,True,42319.990,42331.540,42319.9900,42325.500,<NA>,0,median_ohlc,single_source,True
3,2024-01-01 00:03:00+00:00,BTCUSD,2024,1,True,generated_24_7,42325.50,NaN,NaN,NaN,...,True,42325.500,42368.000,42325.4900,42367.990,<NA>,0,median_ohlc,single_source,True
4,2024-01-01 00:04:00+00:00,BTCUSD,2024,1,True,generated_24_7,42368.00,NaN,NaN,NaN,...,True,42368.000,42397.230,42367.9900,42397.230,<NA>,0,median_ohlc,single_source,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
348410,2024-02-29 23:55:00+00:00,XAUUSD,2024,2,True,interactive_brokers_historical_schedule,NaN,2043.870,2043.91,2043.930,...,True,2043.910,2044.090,2043.8800,2044.040,<NA>,0,median_ohlc,multi_source,True
348411,2024-02-29 23:56:00+00:00,XAUUSD,2024,2,True,interactive_brokers_historical_schedule,NaN,2044.040,2044.04,2044.075,...,True,2044.040,2044.110,2043.9435,2044.030,<NA>,0,median_ohlc,multi_source,True
348412,2024-02-29 23:57:00+00:00,XAUUSD,2024,2,True,interactive_brokers_historical_schedule,NaN,2044.045,2044.03,2044.045,...,True,2044.045,2044.055,2043.9500,2044.015,<NA>,0,median_ohlc,multi_source,True
348413,2024-02-29 23:58:00+00:00,XAUUSD,2024,2,True,interactive_brokers_historical_schedule,NaN,2044.050,2044.00,2044.035,...,True,2044.035,2044.280,2044.0350,2044.280,<NA>,0,median_ohlc,multi_source,True


In [2]:
quality_summary_df = (
    generated_df
    .groupby(
        ["asset", "consensus_quality"],
        dropna=False,
    )
    .agg(
        rows=("time", "count"),
        generated_rows=("is_generated", "sum"),
        avg_broker_count=("broker_count", "mean"),
        max_close_diff_pct=("close_diff_pct", "max"),
    )
    .reset_index()
)

quality_summary_df


,asset,consensus_quality,rows,generated_rows,avg_broker_count,max_close_diff_pct
0,BTCUSD,multi_source,82969,82969,2.000000,0.922055
1,BTCUSD,single_source,3431,3431,1.000000,0.000000
2,DAX,missing,3,0,NaN,NaN
3,DAX,multi_source,21275,21275,2.000000,0.150304
4,DAX,single_source,1082,1082,1.000000,0.000000
5,EURUSD,missing,1459,0,NaN,NaN
6,EURUSD,multi_source,58873,58873,2.917381,0.036078
7,EURUSD,single_source,1243,1243,1.000000,0.000000
8,US500,missing,186,0,NaN,NaN
9,US500,multi_source,50253,50253,2.000000,0.132733
